In [7]:
# Install libraries
!pip install pandas openai tqdm

# Upload file
from google.colab import files
uploaded = files.upload()

# Imports
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

# Add your API key
client = OpenAI(api_key="--API key--")

# Load file
file_name = list(uploaded.keys())[0]
df = pd.read_excel(file_name)

# Function to generate expert response
def generate_expert_response(user_msg, bot_msg):
    prompt = f"""
You are an expert customer service assistant.

A chatbot failed to respond correctly. Your task is to generate a better response.

Rules:
- Be polite and professional
- Use appropriate customer service tone
- Directly address the user's issue
- Fix the chatbot mistake
- Keep the response simple and clear
- Do NOT repeat the chatbot response
- Avoid generic replies
- Avoid repeating sentence structures across responses
- Make each response unique and specific to the case

User Message:
{user_msg}

Chatbot Response (incorrect):
{bot_msg}

Expert Response:
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )
        return response.choices[0].message.content.strip()

    except Exception as e:
        return "Error"

# Generate responses
expert_responses = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    user_msg = row["User_Message"]
    bot_msg = row["Bot_Response"]

    reply = generate_expert_response(user_msg, bot_msg)
    expert_responses.append(reply)

    # Save progress every 5 rows
    if (i + 1) % 5 == 0:
      temp_df = df.iloc[:len(expert_responses)].copy()
      temp_df["Expert_Response"] = expert_responses
      temp_df.to_excel("backup.xlsx", index=False)

# Save final file
df["Expert_Response"] = expert_responses
output_file = "Synth_with_expert_responses.xlsx"
df.to_excel(output_file, index=False)

print("Done!")

# Download file
files.download(output_file)


Saving DBDC_breakdown_caseonly.xlsx to DBDC_breakdown_caseonly.xlsx


100%|██████████| 1019/1019 [16:56<00:00,  1.00it/s]

Done!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>